<a href="https://colab.research.google.com/github/ericrulz643/Insurance-Risk-Pricing-Analysis/blob/main/notebooks/insurance_risk_pricing_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("All libraries loaded successfully!")

# Data Loading and Preparation

This section imports the required Python libraries and loads the motor insurance datasets for analysis.

In [ ]:
freq = pd.read_csv("freMTPL2freq.csv")
sev = pd.read_csv("freMTPL2sev.csv")

print("Frequency dataset:", freq.shape)
print("Severity dataset:", sev.shape)

display(freq.head())
display(sev.head())

In [ ]:
print("FREQUENCY DATA")
print("Columns:", freq.columns.tolist())
print("\nMissing values:")
display(freq.isnull().sum())

print("\nSEVERITY DATA")
print("Columns:", sev.columns.tolist())
print("\nMissing values:")
display(sev.isnull().sum())

In [ ]:
# Summary statistics for numerical variables
print("FREQUENCY DATA SUMMARY")
display(freq.describe().round(2))

print("SEVERITY DATA SUMMARY")
display(sev["ClaimAmount"].describe().round(2))

In [ ]:
# Calculate overall claim frequency and exposure
total_claims = freq["ClaimNb"].sum()
total_exposure = freq["Exposure"].sum()
policies_with_claims = (freq["ClaimNb"] > 0).sum()

claim_frequency = total_claims / total_exposure

print(f"Number of policy records: {len(freq):,}")
print(f"Total claims recorded: {total_claims:,}")
print(f"Total exposure (policy-years): {total_exposure:,.2f}")
print(f"Policies with at least one claim: {policies_with_claims:,}")
print(f"Overall claim frequency: {claim_frequency:.4f} claims per policy-year")
print(f"Equivalent claims per 100 policy-years: {100 * claim_frequency:.2f}")

# Exploratory Data Analysis

This section explores how claim frequency and claim severity vary across policyholder, vehicle, and geographic characteristics. The goal is to identify patterns in the data before developing statistical models.

## Claim Frequency by Driver Age

In [ ]:
# Group drivers into age bands
age_bins = [17, 25, 35, 45, 55, 65, 101]
age_labels = [
    "18–24", "25–34", "35–44",
    "45–54", "55–64", "65+"
]

freq["AgeGroup"] = pd.cut(
    freq["DrivAge"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

# Summarize claims and exposure by age group
age_analysis = (
    freq.groupby("AgeGroup", observed=False)
    .agg(
        Policies=("IDpol", "size"),
        Claims=("ClaimNb", "sum"),
        Exposure=("Exposure", "sum")
    )
)

age_analysis["ClaimFrequency"] = (
    age_analysis["Claims"] / age_analysis["Exposure"]
)

age_analysis["ClaimsPer100PolicyYears"] = (
    100 * age_analysis["ClaimFrequency"]
)

display(age_analysis.round(4))

In [ ]:
plt.figure(figsize=(9, 5))

age_analysis["ClaimsPer100PolicyYears"].plot(
    kind="bar",
    color="steelblue",
    edgecolor="black"
)

plt.title("Claim Frequency by Driver Age")
plt.xlabel("Driver Age Group")
plt.ylabel("Claims per 100 Policy-Years")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## Claim Frequency by Vehicle Age

In [ ]:
# Group vehicles by age
veh_age_bins = [-1, 1, 5, 10, 15, 20, float("inf")]
veh_age_labels = [
    "0–1", "2–5", "6–10", "11–15", "16–20", "21+"
]

freq["VehAgeGroup"] = pd.cut(
    freq["VehAge"],
    bins=veh_age_bins,
    labels=veh_age_labels
)

# Calculate claims and exposure by vehicle age
veh_age_analysis = (
    freq.groupby("VehAgeGroup", observed=False)
    .agg(
        Policies=("IDpol", "size"),
        Claims=("ClaimNb", "sum"),
        Exposure=("Exposure", "sum")
    )
)

veh_age_analysis["ClaimsPer100PolicyYears"] = (
    100 * veh_age_analysis["Claims"] / veh_age_analysis["Exposure"]
)

display(veh_age_analysis.round(3))

## Claim Frequency by Population Density

In [ ]:
# Create population-density groups
freq["DensityGroup"] = pd.qcut(
    freq["Density"],
    q=5,
    duplicates="drop"
)

density_analysis = (
    freq.groupby("DensityGroup", observed=True)
    .agg(
        Policies=("IDpol", "size"),
        Claims=("ClaimNb", "sum"),
        Exposure=("Exposure", "sum")
    )
)

density_analysis["ClaimsPer100PolicyYears"] = (
    100 * density_analysis["Claims"] / density_analysis["Exposure"]
)

display(density_analysis.round(3))

## Overall Claim Severity

In [ ]:
print("Number of individual claims:", len(sev))
print("Total claim amount:", round(sev["ClaimAmount"].sum(), 2))
print("Average claim amount:", round(sev["ClaimAmount"].mean(), 2))
print("Median claim amount:", round(sev["ClaimAmount"].median(), 2))
print("Largest claim amount:", round(sev["ClaimAmount"].max(), 2))

display(sev["ClaimAmount"].describe().round(2))

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(sev["ClaimAmount"], bins=50, edgecolor="black")
plt.xscale("log")

plt.title("Distribution of Individual Claim Amounts")
plt.xlabel("Claim Amount (log scale)")
plt.ylabel("Number of Claims")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

### Claim Severity Without the Largest 1%

In [ ]:
upper_limit = sev["ClaimAmount"].quantile(0.99)

plt.figure(figsize=(9, 5))

plt.hist(
    sev.loc[sev["ClaimAmount"] <= upper_limit, "ClaimAmount"],
    bins=50,
    edgecolor="black"
)

plt.title("Distribution of Claim Amounts (Up to 99th Percentile)")
plt.xlabel("Claim Amount")
plt.ylabel("Number of Claims")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"99th percentile: {upper_limit:,.2f}")

### Financial Impact of the Largest 1% of Claims

In [ ]:
threshold = sev["ClaimAmount"].quantile(0.99)

largest_claims = sev[sev["ClaimAmount"] > threshold]

share_of_total = (
    largest_claims["ClaimAmount"].sum()
    / sev["ClaimAmount"].sum()
    * 100
)

print("Number of claims in the largest 1% group:", len(largest_claims))
print(f"Share of total claim amount: {share_of_total:.2f}%")

# Initial Pure Premium Estimate

This section combines the overall claim frequency and average claim severity to calculate a baseline estimate of expected claim costs per policy-year.

In [ ]:
# Overall claim frequency
total_claims = freq["ClaimNb"].sum()
total_exposure = freq["Exposure"].sum()
claim_frequency = total_claims / total_exposure

# Average claim severity
total_claim_cost = sev["ClaimAmount"].sum()
number_of_severity_records = len(sev)
average_severity = total_claim_cost / number_of_severity_records

# Initial pure premium estimate
pure_premium = claim_frequency * average_severity

print(f"Total claims in frequency data: {total_claims:,.0f}")
print(f"Total exposure: {total_exposure:,.2f} policy-years")
print(f"Claim frequency: {claim_frequency:.4f} claims per policy-year")
print(f"Average claim severity: {average_severity:,.2f}")
print(f"Estimated pure premium: {pure_premium:,.2f} per policy-year")

## Frequency and Severity Data Reconciliation

In [ ]:
print("Claims in frequency data:", freq["ClaimNb"].sum())
print("Severity records:", len(sev))
print("Total severity amount:", round(sev["ClaimAmount"].sum(), 2))
print("Unique policies in severity data:", sev["IDpol"].nunique())

In [ ]:
# Summarize claim counts by policy in the frequency data
print("Policies with at least one claim:",
      (freq["ClaimNb"] > 0).sum())

print("Policies with multiple claims:",
      (freq["ClaimNb"] > 1).sum())

print("Total claims from policies with multiple claims:",
      freq.loc[freq["ClaimNb"] > 1, "ClaimNb"].sum())

# Check the distribution of severity records per policy
severity_records_per_policy = sev.groupby("IDpol").size()

print("\nPolicies with multiple severity records:",
      (severity_records_per_policy > 1).sum())

print("Maximum severity records for one policy:",
      severity_records_per_policy.max())

In [ ]:
# Compare policy IDs across the two datasets
freq_policy_ids = set(freq["IDpol"])
sev_policy_ids = set(sev["IDpol"])

print("Unique policies in frequency data:", len(freq_policy_ids))
print("Unique policies in severity data:", len(sev_policy_ids))

print("Severity policies also found in frequency data:",
      len(sev_policy_ids & freq_policy_ids))

# Check whether severity policies have claims in the frequency data
freq_claims_by_policy = freq.set_index("IDpol")["ClaimNb"]

severity_policy_claims = freq_claims_by_policy.reindex(sev["IDpol"].unique())

print("Severity policies with no claims in frequency data:",
      (severity_policy_claims == 0).sum())

print("Severity policies with at least one claim:",
      (severity_policy_claims > 0).sum())

In [ ]:
# Total number of claims reported by policies present in both files
matched_policy_ids = sev["IDpol"].unique()
matched_claim_counts = (
    freq.loc[freq["IDpol"].isin(matched_policy_ids), "ClaimNb"].sum()
)

print("Severity records:", len(sev))
print("Claims reported by matched policies:", matched_claim_counts)
print("Difference:", matched_claim_counts - len(sev))

# Compare the number of claims and severity records by policy
severity_counts = sev.groupby("IDpol").size().rename("SeverityRecords")

policy_comparison = (
    freq.set_index("IDpol")[["ClaimNb"]]
    .join(severity_counts, how="inner")
)

print("\nPolicies where claim count differs from severity-record count:",
      (policy_comparison["ClaimNb"] != policy_comparison["SeverityRecords"]).sum())

print("\nFirst 10 policies with mismatched claim counts:")
print(
    policy_comparison.loc[
        policy_comparison["ClaimNb"] != policy_comparison["SeverityRecords"]
    ].head(10)
)

### Detailed Claim-Count Reconciliation

In [ ]:
# Compare totals using the same joined table
print("Frequency claim total for matched policies:",
      policy_comparison["ClaimNb"].sum())

print("Severity record total for matched policies:",
      policy_comparison["SeverityRecords"].sum())

print("Number of policies compared:",
      len(policy_comparison))

print("Missing claim counts:",
      policy_comparison["ClaimNb"].isna().sum())

print("Missing severity counts:",
      policy_comparison["SeverityRecords"].isna().sum())

# Show all policies with unequal counts, including missing values
mismatches = policy_comparison[
    policy_comparison["ClaimNb"].ne(policy_comparison["SeverityRecords"])
]

print("\nNumber of mismatches:", len(mismatches))
print(mismatches.head(20))

### Investigation of Unmatched Severity Policies

In [ ]:
# Severity policy IDs not found in the frequency data
unmatched_ids = sev.loc[
    ~sev["IDpol"].isin(freq["IDpol"]), "IDpol"
].unique()

print("Unmatched severity policy IDs:", unmatched_ids)

print("\nSeverity records for unmatched policies:")
print(
    sev[sev["IDpol"].isin(unmatched_ids)]
    .sort_values("IDpol")
)

# Claim Frequency Modeling

This section prepares the policy-level data for statistical modeling. Driver age, vehicle age, and population density are transformed into variables that can be used to estimate claim frequency.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Keep policies with positive exposure
model_data = freq.loc[freq["Exposure"] > 0].copy()

# Create age groups
model_data["DriverAgeGroup"] = pd.cut(
    model_data["DrivAge"],
    bins=[17, 25, 35, 45, 55, 65, float("inf")],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False
)

# Create vehicle age groups
model_data["VehicleAgeGroup"] = pd.cut(
    model_data["VehAge"],
    bins=[-1, 1, 5, 10, 15, 20, float("inf")],
    labels=["0-1", "2-5", "6-10", "11-15", "16-20", "21+"]
)

# Use log population density to reduce its extreme right skew
model_data["LogDensity"] = np.log1p(model_data["Density"])

print("Original policy records:", len(freq))
print("Records used in model:", len(model_data))
print("Total claims in model:", model_data["ClaimNb"].sum())
print("Total exposure in model:", round(model_data["Exposure"].sum(), 2))
print("Missing values in modeling variables:")
print(model_data[
    ["ClaimNb", "Exposure", "DriverAgeGroup", "VehicleAgeGroup", "LogDensity"]
].isna().sum())

## Initial Poisson Regression Model

In [ ]:
# Fit a Poisson regression with an exposure offset
poisson_model = smf.glm(
    formula=(
        "ClaimNb ~ C(DriverAgeGroup) "
        "+ C(VehicleAgeGroup) "
        "+ LogDensity"
    ),
    data=model_data,
    family=sm.families.Poisson(),
    offset=np.log(model_data["Exposure"])
).fit()

print(poisson_model.summary())

## Model Interpretation and Overdispersion

In [ ]:
# Convert model coefficients to rate ratios
rate_ratios = np.exp(poisson_model.params)

results = pd.DataFrame({
    "Coefficient": poisson_model.params,
    "Rate Ratio": rate_ratios,
    "Percent Change": (rate_ratios - 1) * 100,
    "P-value": poisson_model.pvalues
})

print(results.round(4))

# Check for overdispersion
pearson_dispersion = (
    poisson_model.pearson_chi2 / poisson_model.df_resid
)

print(f"\nPearson dispersion ratio: {pearson_dispersion:.3f}")

### Alternative: Negative Binomial Regression

In [ ]:
negative_binomial_model = smf.glm(
    formula=(
        "ClaimNb ~ C(DriverAgeGroup) "
        "+ C(VehicleAgeGroup) "
        "+ LogDensity"
    ),
    data=model_data,
    family=sm.families.NegativeBinomial(alpha=1.0),
    offset=np.log(model_data["Exposure"])
).fit()

print(negative_binomial_model.summary())

### Poisson Regression with Robust Standard Errors

In [ ]:
poisson_robust = smf.glm(
    formula=(
        "ClaimNb ~ C(DriverAgeGroup) "
        "+ C(VehicleAgeGroup) "
        "+ LogDensity"
    ),
    data=model_data,
    family=sm.families.Poisson(),
    offset=np.log(model_data["Exposure"])
).fit(cov_type="HC0")

print(poisson_robust.summary())

### Robust Poisson Model: Rate Ratios and Confidence Intervals

In [ ]:
params = poisson_robust.params
conf = poisson_robust.conf_int()

results_table = pd.DataFrame({
    "Coefficient": params,
    "Rate Ratio": np.exp(params),
    "RR 95% CI Lower": np.exp(conf[0]),
    "RR 95% CI Upper": np.exp(conf[1]),
    "P-value": poisson_robust.pvalues
})

results_table = results_table.drop("Intercept")

results_table = results_table.round({
    "Coefficient": 4,
    "Rate Ratio": 3,
    "RR 95% CI Lower": 3,
    "RR 95% CI Upper": 3,
    "P-value": 4
})

results_table

### Risk Segmentation and Decile Validation

In [ ]:
# Calculate predicted claim counts for each policy
model_data["PredictedClaims"] = poisson_robust.predict(
    model_data,
    offset=np.log(model_data["Exposure"])
)

# Rank policies into 10 groups based on predicted claim rate
model_data["RiskDecile"] = pd.qcut(
    model_data["PredictedClaims"] / model_data["Exposure"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

# Compare actual and predicted frequency by risk decile
validation_table = (
    model_data.groupby("RiskDecile", observed=True)
    .agg(
        Policies=("ClaimNb", "size"),
        ActualClaims=("ClaimNb", "sum"),
        PredictedClaims=("PredictedClaims", "sum"),
        Exposure=("Exposure", "sum")
    )
    .reset_index()
)

validation_table["ActualFrequency"] = (
    validation_table["ActualClaims"] / validation_table["Exposure"]
)

validation_table["PredictedFrequency"] = (
    validation_table["PredictedClaims"] / validation_table["Exposure"]
)

validation_table["ActualClaimsPer100Years"] = (
    100 * validation_table["ActualFrequency"]
)

validation_table["PredictedClaimsPer100Years"] = (
    100 * validation_table["PredictedFrequency"]
)

validation_table.round(3)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    validation_table["RiskDecile"],
    validation_table["ActualClaimsPer100Years"],
    marker="o",
    label="Actual frequency"
)

plt.plot(
    validation_table["RiskDecile"],
    validation_table["PredictedClaimsPer100Years"],
    marker="s",
    linestyle="--",
    label="Predicted frequency"
)

plt.xlabel("Predicted Risk Decile (1 = Lowest Risk)")
plt.ylabel("Claims per 100 Policy-Years")
plt.title("Actual vs. Predicted Claim Frequency by Risk Decile")
plt.xticks(range(1, 11))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Five-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Number of records:", len(model_data))
print("Number of folds:", kf.get_n_splits())

In [ ]:
from sklearn.metrics import mean_poisson_deviance

formula = (
    "ClaimNb ~ C(DriverAgeGroup) "
    "+ C(VehicleAgeGroup) "
    "+ LogDensity"
)

cv_results = []

for fold, (train_idx, test_idx) in enumerate(
    kf.split(model_data), start=1
):
    train = model_data.iloc[train_idx]
    test = model_data.iloc[test_idx]

    # Fit model using training data only
    model = smf.glm(
        formula=formula,
        data=train,
        family=sm.families.Poisson(),
        offset=np.log(train["Exposure"])
    ).fit()

    # Predict claim counts for held-out policies
    predicted = model.predict(
        test,
        offset=np.log(test["Exposure"])
    )

    # Calculate out-of-sample performance
    deviance = mean_poisson_deviance(
        test["ClaimNb"],
        predicted / test["Exposure"]
    )

    actual_frequency = (
        test["ClaimNb"].sum() / test["Exposure"].sum()
    )

    predicted_frequency = (
        predicted.sum() / test["Exposure"].sum()
    )

    cv_results.append({
        "Fold": fold,
        "TestRecords": len(test),
        "ActualFrequency": actual_frequency,
        "PredictedFrequency": predicted_frequency,
        "MeanPoissonDeviance": deviance
    })

cv_table = pd.DataFrame(cv_results)
cv_table.round(5)

### Model Performance Compared with a Baseline

In [ ]:
baseline_results = []

for fold, (train_idx, test_idx) in enumerate(
    kf.split(model_data), start=1
):
    train = model_data.iloc[train_idx]
    test = model_data.iloc[test_idx]

    # Estimate one overall claim rate from training data
    overall_rate = train["ClaimNb"].sum() / train["Exposure"].sum()

    # Predict each test policy's claim count using its exposure
    baseline_pred = overall_rate * test["Exposure"]

    # Fit the risk-factor model using training data only
    model = smf.glm(
        formula=formula,
        data=train,
        family=sm.families.Poisson(),
        offset=np.log(train["Exposure"])
    ).fit()

    model_pred = model.predict(
        test,
        offset=np.log(test["Exposure"])
    )

    baseline_deviance = mean_poisson_deviance(
        test["ClaimNb"], baseline_pred
    )

    model_deviance = mean_poisson_deviance(
        test["ClaimNb"], model_pred
    )

    baseline_results.append({
        "Fold": fold,
        "BaselineDeviance": baseline_deviance,
        "GLMDeviance": model_deviance
    })

baseline_table = pd.DataFrame(baseline_results)

baseline_table["ImprovementPercent"] = (
    100 * (
        baseline_table["BaselineDeviance"]
        - baseline_table["GLMDeviance"]
    ) / baseline_table["BaselineDeviance"]
)

print(baseline_table.round(5))

print("\nAverage baseline deviance:",
      round(baseline_table["BaselineDeviance"].mean(), 5))

print("Average GLM deviance:",
      round(baseline_table["GLMDeviance"].mean(), 5))

print("Average deviance improvement (%):",
      round(baseline_table["ImprovementPercent"].mean(), 2))

### Testing Additional Risk Factors

In [ ]:
expanded_formula = (
    "ClaimNb ~ C(DriverAgeGroup) "
    "+ C(VehicleAgeGroup) "
    "+ LogDensity "
    "+ BonusMalus"
)

expanded_cv_results = []

for fold, (train_idx, test_idx) in enumerate(
    kf.split(model_data), start=1
):
    train = model_data.iloc[train_idx]
    test = model_data.iloc[test_idx]

    model = smf.glm(
        formula=expanded_formula,
        data=train,
        family=sm.families.Poisson(),
        offset=np.log(train["Exposure"])
    ).fit()

    predicted = model.predict(
        test,
        offset=np.log(test["Exposure"])
    )

    deviance = mean_poisson_deviance(
        test["ClaimNb"], predicted
    )

    expanded_cv_results.append(deviance)

print("Expanded model average CV deviance:",
      round(np.mean(expanded_cv_results), 5))
print("Expanded model CV standard deviation:",
      round(np.std(expanded_cv_results, ddof=1), 5))
print("Previous model average CV deviance: 0.32535")

In [ ]:
expanded_formula_2 = (
    "ClaimNb ~ C(DriverAgeGroup) "
    "+ C(VehicleAgeGroup) "
    "+ LogDensity "
    "+ BonusMalus "
    "+ C(VehPower)"
)

vehpower_cv_results = []

for fold, (train_idx, test_idx) in enumerate(
    kf.split(model_data), start=1
):
    train = model_data.iloc[train_idx]
    test = model_data.iloc[test_idx]

    model = smf.glm(
        formula=expanded_formula_2,
        data=train,
        family=sm.families.Poisson(),
        offset=np.log(train["Exposure"])
    ).fit()

    predicted = model.predict(
        test,
        offset=np.log(test["Exposure"])
    )

    deviance = mean_poisson_deviance(
        test["ClaimNb"], predicted
    )

    vehpower_cv_results.append(deviance)

print("Model with BonusMalus and VehPower:")
print("Average CV deviance:",
      round(np.mean(vehpower_cv_results), 5))
print("CV standard deviation:",
      round(np.std(vehpower_cv_results, ddof=1), 5))
print("Previous model average CV deviance: 0.31958")

In [ ]:
expanded_formula_3 = (
    "ClaimNb ~ C(DriverAgeGroup) "
    "+ C(VehicleAgeGroup) "
    "+ LogDensity "
    "+ BonusMalus "
    "+ C(VehPower) "
    "+ C(VehBrand)"
)

brand_cv_results = []

for fold, (train_idx, test_idx) in enumerate(
    kf.split(model_data), start=1
):
    train = model_data.iloc[train_idx]
    test = model_data.iloc[test_idx]

    model = smf.glm(
        formula=expanded_formula_3,
        data=train,
        family=sm.families.Poisson(),
        offset=np.log(train["Exposure"])
    ).fit()

    predicted = model.predict(
        test,
        offset=np.log(test["Exposure"])
    )

    deviance = mean_poisson_deviance(
        test["ClaimNb"], predicted
    )

    brand_cv_results.append(deviance)

print("Model with BonusMalus, VehPower, and VehBrand:")
print("Average CV deviance:",
      round(np.mean(brand_cv_results), 5))
print("CV standard deviation:",
      round(np.std(brand_cv_results, ddof=1), 5))
print("Previous model average CV deviance: 0.31922")

In [ ]:
expanded_formula_4 = (
    "ClaimNb ~ C(DriverAgeGroup) "
    "+ C(VehicleAgeGroup) "
    "+ LogDensity "
    "+ BonusMalus "
    "+ C(VehPower) "
    "+ C(VehBrand) "
    "+ C(Area)"
)

area_cv_results = []

for fold, (train_idx, test_idx) in enumerate(
    kf.split(model_data), start=1
):
    train = model_data.iloc[train_idx]
    test = model_data.iloc[test_idx]

    model = smf.glm(
        formula=expanded_formula_4,
        data=train,
        family=sm.families.Poisson(),
        offset=np.log(train["Exposure"])
    ).fit()

    predicted = model.predict(
        test,
        offset=np.log(test["Exposure"])
    )

    deviance = mean_poisson_deviance(
        test["ClaimNb"], predicted
    )

    area_cv_results.append(deviance)

print("Model including Area:")
print("Average CV deviance:",
      round(np.mean(area_cv_results), 5))
print("CV standard deviation:",
      round(np.std(area_cv_results, ddof=1), 5))
print("Previous model average CV deviance:",
      round(np.mean(brand_cv_results), 5))

In [ ]:
model_data = freq.loc[freq["Exposure"] > 0].copy()

model_data["DriverAgeGroup"] = pd.cut(
    model_data["DrivAge"],
    bins=[17, 25, 35, 45, 55, 65, float("inf")],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False
)

model_data["VehicleAgeGroup"] = pd.cut(
    model_data["VehAge"],
    bins=[-1, 1, 5, 10, 15, 20, float("inf")],
    labels=["0-1", "2-5", "6-10", "11-15", "16-20", "21+"]
)

model_data["LogDensity"] = np.log1p(model_data["Density"])

### Model Development on a Random Sample

In [ ]:
model_sample = model_data.sample(
    n=150_000,
    random_state=42
).copy()

kf_sample = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Sample size:", len(model_sample))

In [ ]:
base_formula = (
    "ClaimNb ~ C(DriverAgeGroup) "
    "+ C(VehicleAgeGroup) "
    "+ LogDensity "
    "+ BonusMalus "
    "+ C(VehPower) "
    "+ C(VehBrand)"
)

area_formula = base_formula + " + C(Area)"

comparison_results = []

for name, formula_test in [
    ("Without Area", base_formula),
    ("With Area", area_formula)
]:
    fold_scores = []

    for train_idx, test_idx in kf_sample.split(model_sample):
        train = model_sample.iloc[train_idx]
        test = model_sample.iloc[test_idx]

        fitted_model = smf.glm(
            formula=formula_test,
            data=train,
            family=sm.families.Poisson(),
            offset=np.log(train["Exposure"])
        ).fit()

        predictions = fitted_model.predict(
            test,
            offset=np.log(test["Exposure"])
        )

        fold_scores.append(
            mean_poisson_deviance(
                test["ClaimNb"], predictions
            )
        )

    comparison_results.append({
        "Model": name,
        "Average CV Deviance": np.mean(fold_scores),
        "CV Standard Deviation": np.std(fold_scores, ddof=1)
    })

print(pd.DataFrame(comparison_results).round(5).to_string(index=False))

In [ ]:
region_formula = base_formula + " + C(Region)"

region_comparison = []

for name, formula_test in [
    ("Without Region", base_formula),
    ("With Region", region_formula)
]:
    fold_scores = []

    for train_idx, test_idx in kf_sample.split(model_sample):
        train = model_sample.iloc[train_idx]
        test = model_sample.iloc[test_idx]

        fitted_model = smf.glm(
            formula=formula_test,
            data=train,
            family=sm.families.Poisson(),
            offset=np.log(train["Exposure"])
        ).fit()

        predictions = fitted_model.predict(
            test,
            offset=np.log(test["Exposure"])
        )

        fold_scores.append(
            mean_poisson_deviance(
                test["ClaimNb"], predictions
            )
        )

    region_comparison.append({
        "Model": name,
        "Average CV Deviance": np.mean(fold_scores),
        "CV Standard Deviation": np.std(fold_scores, ddof=1)
    })

print(pd.DataFrame(region_comparison).round(5).to_string(index=False))

results_df = pd.DataFrame(comparison_results)

print(
    "\nChange in average CV deviance:",
    round(
        results_df.loc[1, "Average CV Deviance"]
        - results_df.loc[0, "Average CV Deviance"],
        5
    )
)

In [ ]:
gas_formula = base_formula + " + C(VehGas)"

gas_comparison = []

for name, formula_test in [
    ("Without VehGas", base_formula),
    ("With VehGas", gas_formula)
]:
    fold_scores = []

    for train_idx, test_idx in kf_sample.split(model_sample):
        train = model_sample.iloc[train_idx]
        test = model_sample.iloc[test_idx]

        fitted_model = smf.glm(
            formula=formula_test,
            data=train,
            family=sm.families.Poisson(),
            offset=np.log(train["Exposure"])
        ).fit()

        predictions = fitted_model.predict(
            test,
            offset=np.log(test["Exposure"])
        )

        fold_scores.append(
            mean_poisson_deviance(
                test["ClaimNb"], predictions
            )
        )

    gas_comparison.append({
        "Model": name,
        "Average CV Deviance": np.mean(fold_scores),
        "CV Standard Deviation": np.std(fold_scores, ddof=1)
    })

print(pd.DataFrame(gas_comparison).round(5).to_string(index=False))

### Final Frequency Model on the Sample

In [ ]:
final_formula = (
    "ClaimNb ~ C(DriverAgeGroup) "
    "+ C(VehicleAgeGroup) "
    "+ LogDensity "
    "+ BonusMalus "
    "+ C(VehPower) "
    "+ C(VehBrand) "
    "+ C(VehGas)"
)

final_model = smf.glm(
    formula=final_formula,
    data=model_sample,
    family=sm.families.Poisson(),
    offset=np.log(model_sample["Exposure"])
).fit(cov_type="HC0")

print(final_model.summary())

In [ ]:
# Convert coefficients and confidence intervals to rate ratios
coef = final_model.params
ci = final_model.conf_int()

rate_ratio_table = pd.DataFrame({
    "Coefficient": coef,
    "Rate Ratio": np.exp(coef),
    "95% CI Lower": np.exp(ci[0]),
    "95% CI Upper": np.exp(ci[1]),
    "P-value": final_model.pvalues
})

# Exclude the intercept and format the results
rate_ratio_table = rate_ratio_table.drop("Intercept")
rate_ratio_table["P-value"] = rate_ratio_table["P-value"].map(
    lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
)

print(rate_ratio_table.round(3).to_string())

### Out-of-Fold Prediction and Validation

In [ ]:
# Create a validation table using out-of-fold predictions
validation_data = model_sample[
    ["ClaimNb", "Exposure"]
].copy()

validation_data["PredictedClaims"] = oof_predictions

validation_data["RiskDecile"] = pd.qcut(
    validation_data["PredictedClaims"].rank(method="first"),
    q=10,
    labels=False
) + 1

decile_validation = (
    validation_data
    .groupby("RiskDecile", observed=True)
    .agg(
        Policies=("ClaimNb", "size"),
        ActualClaims=("ClaimNb", "sum"),
        PredictedClaims=("PredictedClaims", "sum"),
        Exposure=("Exposure", "sum")
    )
    .reset_index()
)

decile_validation["ActualFreqPer100"] = (
    100 * decile_validation["ActualClaims"]
    / decile_validation["Exposure"]
)

decile_validation["PredictedFreqPer100"] = (
    100 * decile_validation["PredictedClaims"]
    / decile_validation["Exposure"]
)

print(decile_validation.round(3).to_string(index=False))

In [ ]:
validation_data = model_sample[
    ["ClaimNb", "Exposure"]
].copy()

validation_data["PredictedClaims"] = oof_predictions

# Estimate claim frequency per policy-year
validation_data["PredictedRate"] = (
    validation_data["PredictedClaims"]
    / validation_data["Exposure"]
)

# Rank policies by predicted frequency, not predicted claim count
validation_data["RiskDecile"] = pd.qcut(
    validation_data["PredictedRate"].rank(method="first"),
    q=10,
    labels=False
) + 1

decile_validation = (
    validation_data
    .groupby("RiskDecile", observed=True)
    .agg(
        Policies=("ClaimNb", "size"),
        ActualClaims=("ClaimNb", "sum"),
        PredictedClaims=("PredictedClaims", "sum"),
        Exposure=("Exposure", "sum")
    )
    .reset_index()
)

decile_validation["ActualFreqPer100"] = (
    100 * decile_validation["ActualClaims"]
    / decile_validation["Exposure"]
)

decile_validation["PredictedFreqPer100"] = (
    100 * decile_validation["PredictedClaims"]
    / decile_validation["Exposure"]
)

print(decile_validation.round(3).to_string(index=False))

In [ ]:
oof_predictions = np.zeros(len(model_sample))

for train_idx, test_idx in kf_sample.split(model_sample):
    train = model_sample.iloc[train_idx]
    test = model_sample.iloc[test_idx]

    fold_model = smf.glm(
        formula=final_formula,
        data=train,
        family=sm.families.Poisson(),
        offset=np.log(train["Exposure"])
    ).fit()

    oof_predictions[test_idx] = fold_model.predict(
        test,
        offset=np.log(test["Exposure"])
    )

print("Policies evaluated:", len(oof_predictions))
print("Predictions generated:", np.isfinite(oof_predictions).sum())
print("Average predicted claims per policy:", oof_predictions.mean())

In [ ]:
actual_total = model_sample["ClaimNb"].sum()
predicted_total = oof_predictions.sum()

print(f"Actual claims: {actual_total:.2f}")
print(f"Predicted claims: {predicted_total:.2f}")
print(
    "Relative prediction error: "
    f"{100 * (predicted_total - actual_total) / actual_total:.2f}%"
)

# Claim Severity Analysis

In [ ]:
print("Severity dataset shape:", sev.shape)
print("\nColumn names:")
print(sev.columns.tolist())

print("\nFirst five rows:")
display(sev.head())

print("\nSummary statistics:")
display(sev["ClaimAmount"].describe())

print("\nMissing values:")
print(sev.isnull().sum())

## Claim Severity Distribution and Large-Loss Analysis

In [ ]:
total_claim_amount = sev["ClaimAmount"].sum()

percentiles = sev["ClaimAmount"].quantile(
    [0.90, 0.95, 0.99, 0.995]
)

print(f"Number of claims: {len(sev):,}")
print(f"Total claim amount: {total_claim_amount:,.2f}")
print(f"Mean claim amount: {sev['ClaimAmount'].mean():,.2f}")
print(f"Median claim amount: {sev['ClaimAmount'].median():,.2f}")

print("\nClaim amount percentiles:")
print(percentiles.round(2).to_string())

print("\nContribution of largest claims:")
for percentile in [0.90, 0.95, 0.99, 0.995]:
    threshold = sev["ClaimAmount"].quantile(percentile)
    largest_claims = sev.loc[
        sev["ClaimAmount"] >= threshold, "ClaimAmount"
    ]

    share = 100 * largest_claims.sum() / total_claim_amount

    print(
        f"Top {100 * (1 - percentile):.1f}% of claims: "
        f"{len(largest_claims):,} claims, "
        f"{share:.2f}% of total claim amount"
    )

In [ ]:
# Original scale
plt.figure(figsize=(10, 5))
plt.hist(sev["ClaimAmount"], bins=100)
plt.title("Distribution of Claim Amounts")
plt.xlabel("Claim Amount")
plt.ylabel("Number of Claims")
plt.tight_layout()
plt.show()

# Logarithmic scale
plt.figure(figsize=(10, 5))
plt.hist(sev["ClaimAmount"], bins=100)
plt.xscale("log")
plt.title("Distribution of Claim Amounts (Log Scale)")
plt.xlabel("Claim Amount — logarithmic scale")
plt.ylabel("Number of Claims")
plt.tight_layout()
plt.show()

In [ ]:
claim_amounts = sev["ClaimAmount"]

# Create bin edges evenly spaced on a logarithmic scale
log_bins = np.logspace(
    np.log10(claim_amounts.min()),
    np.log10(claim_amounts.max()),
    60
)

plt.figure(figsize=(10, 5))
plt.hist(claim_amounts, bins=log_bins)
plt.xscale("log")
plt.title("Distribution of Claim Amounts (Log-Spaced Bins)")
plt.xlabel("Claim Amount (log scale)")
plt.ylabel("Number of Claims")
plt.tight_layout()
plt.show()

## Baseline Severity Model

In [ ]:
severity_model = sm.GLM(
    sev["ClaimAmount"],
    np.ones((len(sev), 1)),
    family=sm.families.Gamma(
        link=sm.families.links.Log()
    )
).fit()

print(severity_model.summary())

### Sensitivity to Large Claims

In [ ]:
claim_amounts = sev["ClaimAmount"]

threshold_99 = claim_amounts.quantile(0.99)

trimmed_claims = claim_amounts[
    claim_amounts <= threshold_99
]

print(f"99th percentile threshold: {threshold_99:,.2f}")
print(f"Original mean: {claim_amounts.mean():,.2f}")
print(f"Mean excluding claims above the 99th percentile: "
      f"{trimmed_claims.mean():,.2f}")
print(f"Original median: {claim_amounts.median():,.2f}")
print(f"Number of claims excluded: "
      f"{len(claim_amounts) - len(trimmed_claims):,}")

In [ ]:
### Lognormal Distribution Comparison

In [ ]:
log_claims = np.log(sev["ClaimAmount"])

# Fit a normal distribution to log claim amounts
log_mean = log_claims.mean()
log_std = log_claims.std(ddof=1)

# Convert lognormal parameters back to the claim-amount scale
lognormal_median = np.exp(log_mean)
lognormal_mean = np.exp(log_mean + 0.5 * log_std**2)

print(f"Mean of log claim amounts: {log_mean:.4f}")
print(f"Standard deviation of log claim amounts: {log_std:.4f}")
print(f"Implied lognormal median: {lognormal_median:,.2f}")
print(f"Implied lognormal mean: {lognormal_mean:,.2f}")
print(f"Observed median: {sev['ClaimAmount'].median():,.2f}")
print(f"Observed mean: {sev['ClaimAmount'].mean():,.2f}")

In [ ]:
from scipy.stats import norm

log_claims = np.log(sev["ClaimAmount"])

plt.figure(figsize=(10, 5))

plt.hist(
    log_claims,
    bins=60,
    density=True,
    alpha=0.6,
    label="Observed log claim amounts"
)

x = np.linspace(log_claims.min(), log_claims.max(), 400)

plt.plot(
    x,
    norm.pdf(x, loc=log_mean, scale=log_std),
    linewidth=2,
    label="Fitted normal density"
)

plt.title("Observed vs. Fitted Lognormal Severity Distribution")
plt.xlabel("Log claim amount")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import lognorm

# Fit a lognormal distribution to the claim amounts
shape, loc, scale = lognorm.fit(sev["ClaimAmount"], floc=0)

percentiles = [50, 75, 90, 95, 99, 99.5, 99.9]

comparison = pd.DataFrame({
    "Percentile": percentiles,
    "Observed": np.percentile(sev["ClaimAmount"], percentiles),
    "Fitted Lognormal": lognorm.ppf(
        np.array(percentiles) / 100,
        shape,
        loc=loc,
        scale=scale
    )
})

comparison["Difference (%)"] = (
    100 * (
        comparison["Fitted Lognormal"] - comparison["Observed"]
    ) / comparison["Observed"]
)

comparison.round(2)

In [ ]:
### Gamma vs. Lognormal Distribution Comparison

In [ ]:
from scipy.stats import gamma, lognorm

claims = sev["ClaimAmount"].to_numpy()

# Fit Gamma distribution
gamma_shape, gamma_loc, gamma_scale = gamma.fit(claims, floc=0)

# Fit lognormal distribution
lognorm_shape, lognorm_loc, lognorm_scale = lognorm.fit(
    claims, floc=0
)

percentiles = np.array([50, 75, 90, 95, 99, 99.5, 99.9])
probabilities = percentiles / 100

comparison = pd.DataFrame({
    "Percentile": percentiles,
    "Observed": np.percentile(claims, percentiles),
    "Gamma": gamma.ppf(
        probabilities, gamma_shape, loc=gamma_loc, scale=gamma_scale
    ),
    "Lognormal": lognorm.ppf(
        probabilities, lognorm_shape, loc=lognorm_loc, scale=lognorm_scale
    )
})

comparison.round(2)

In [ ]:
### Largest Individual Claims

In [ ]:
largest_claims = (
    sev["ClaimAmount"]
    .nlargest(10)
    .reset_index(drop=True)
    .to_frame(name="ClaimAmount")
)

largest_claims.index = largest_claims.index + 1
largest_claims.index.name = "Rank"

largest_claims

### Large Claims and Upper-Tail Analysis

In [ ]:
claims_sorted = np.sort(sev["ClaimAmount"].to_numpy())
n = len(claims_sorted)

# Proportion of claims greater than or equal to each amount
survival_prob = (n - np.arange(n)) / n

plt.figure(figsize=(10, 5))
plt.plot(claims_sorted, survival_prob)
plt.xscale("log")
plt.yscale("log")
plt.title("Claim Severity: Upper-Tail Distribution")
plt.xlabel("Claim amount ($, log scale)")
plt.ylabel("Proportion of claims at or above amount")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### Most Frequently Occurring Claim Amounts

In [ ]:
# Identify the most frequently occurring claim amounts
claim_amount_counts = (
    sev["ClaimAmount"]
    .value_counts()
    .rename_axis("ClaimAmount")
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

claim_amount_counts.head(10)

In [ ]:
# Examine the three most frequent exact claim amounts
top_amounts = sev["ClaimAmount"].value_counts().head(3).index

top_amount_summary = (
    sev[sev["ClaimAmount"].isin(top_amounts)]
    .groupby("ClaimAmount")
    .agg(
        ClaimCount=("ClaimAmount", "size"),
        TotalClaimCost=("ClaimAmount", "sum")
    )
    .reset_index()
)

top_amount_summary["ShareOfClaims (%)"] = (
    100 * top_amount_summary["ClaimCount"] / len(sev)
)

top_amount_summary["ShareOfTotalCost (%)"] = (
    100 * top_amount_summary["TotalClaimCost"] / sev["ClaimAmount"].sum()
)

top_amount_summary.round(2)

In [ ]:
from scipy.stats import gamma, lognorm

claims = sev["ClaimAmount"].to_numpy()

# Fit both distributions, fixing location at zero
gamma_params = gamma.fit(claims, floc=0)
lognorm_params = lognorm.fit(claims, floc=0)

# Calculate log-likelihood on the original claim-amount scale
gamma_loglik = np.sum(gamma.logpdf(claims, *gamma_params))
lognorm_loglik = np.sum(lognorm.logpdf(claims, *lognorm_params))

# Each model estimates two free parameters when loc is fixed at zero
k = 2

comparison = pd.DataFrame({
    "Distribution": ["Gamma", "Lognormal"],
    "Log-Likelihood": [gamma_loglik, lognorm_loglik],
    "AIC": [
        2 * k - 2 * gamma_loglik,
        2 * k - 2 * lognorm_loglik
    ]
})

comparison.round(2)

In [ ]:
from scipy.stats import lognorm

claims = sev["ClaimAmount"].to_numpy()

# Fit the lognormal distribution, fixing location at zero
shape, loc, scale = lognorm.fit(claims, floc=0)

# Calculate matching theoretical and observed quantiles
probabilities = np.linspace(0.001, 0.999, 500)

observed_quantiles = np.quantile(claims, probabilities)
fitted_quantiles = lognorm.ppf(
    probabilities, shape, loc=loc, scale=scale
)

plt.figure(figsize=(8, 8))
plt.scatter(fitted_quantiles, observed_quantiles, s=12)

# Reference line: perfect agreement
upper = max(observed_quantiles.max(), fitted_quantiles.max())
plt.plot([0, upper], [0, upper], linestyle="--")

plt.xscale("log")
plt.yscale("log")
plt.title("Q–Q Plot: Observed vs. Fitted Lognormal Severity")
plt.xlabel("Fitted Lognormal Quantiles ($)")
plt.ylabel("Observed Claim Quantiles ($)")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## Pure Premium Analysis by Driver Age

In [ ]:
# Attach driver age group to each policy in the frequency data
policy_ages = model_data[["IDpol", "DriverAgeGroup"]].drop_duplicates()

# Match each individual claim to its policy's driver age group
severity_by_age = sev.merge(
    policy_ages,
    on="IDpol",
    how="inner"
)

# Calculate frequency by driver age group
frequency_by_age = (
    model_data.groupby("DriverAgeGroup", observed=True)
    .agg(
        Policies=("IDpol", "size"),
        TotalClaims=("ClaimNb", "sum"),
        Exposure=("Exposure", "sum")
    )
)

frequency_by_age["ClaimsPer100PolicyYears"] = (
    100 * frequency_by_age["TotalClaims"]
    / frequency_by_age["Exposure"]
)

# Calculate average severity by driver age group
severity_summary = (
    severity_by_age.groupby("DriverAgeGroup", observed=True)
    .agg(
        MatchedClaimCount=("ClaimAmount", "size"),
        TotalClaimCost=("ClaimAmount", "sum"),
        AverageSeverity=("ClaimAmount", "mean")
    )
)

# Combine frequency and severity estimates
pure_premium = frequency_by_age.join(severity_summary)

pure_premium["PurePremium"] = (
    pure_premium["TotalClaims"]
    / pure_premium["Exposure"]
    * pure_premium["AverageSeverity"]
)

pure_premium[
    [
        "Policies",
        "ClaimsPer100PolicyYears",
        "MatchedClaimCount",
        "AverageSeverity",
        "PurePremium"
    ]
].round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pure_premium["ClaimsPer100PolicyYears"].plot(
    kind="bar", ax=axes[0]
)
axes[0].set_title("Claim Frequency by Driver Age")
axes[0].set_ylabel("Claims per 100 policy-years")
axes[0].set_xlabel("Driver age group")
axes[0].tick_params(axis="x", rotation=45)

pure_premium["AverageSeverity"].plot(
    kind="bar", ax=axes[1]
)
axes[1].set_title("Average Claim Severity by Driver Age")
axes[1].set_ylabel("Average claim amount")
axes[1].set_xlabel("Driver age group")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Claim Frequency Modeling

In [ ]:
model_data = freq.loc[freq["Exposure"] > 0].copy()

model_data["DriverAgeGroup"] = pd.cut(
    model_data["DrivAge"],
    bins=[17, 25, 35, 45, 55, 65, float("inf")],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False
)

model_data["VehicleAgeGroup"] = pd.cut(
    model_data["VehAge"],
    bins=[-1, 1, 5, 10, 15, 20, float("inf")],
    labels=["0-1", "2-5", "6-10", "11-15", "16-20", "21+"]
)

model_data["LogDensity"] = np.log1p(model_data["Density"])

print("Modeling dataset shape:", model_data.shape)
print("Missing values:", model_data.isnull().sum().sum())

In [ ]:
model_sample = model_data.sample(
    n=150_000,
    random_state=42
).copy()

kf_sample = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Sample size:", len(model_sample))

In [ ]:
print("Frequency records:", len(freq))
print("Severity records:", len(sev))
print("Modeling records:", len(model_data))
print("Modeling sample:", len(model_sample))

print("\nFrequency columns:")
print(freq.columns.tolist())

print("\nSeverity columns:")
print(sev.columns.tolist())

## Link Severity Records to Driver Age

In [ ]:
# Get each policy's driver age group
policy_ages = model_data[
    ["IDpol", "DriverAgeGroup"]
].drop_duplicates()

# Match individual claims to driver age groups
severity_by_age = sev.merge(
    policy_ages,
    on="IDpol",
    how="inner"
)

print("Matched severity records:", len(severity_by_age))
print("\nClaims by driver age group:")
print(
    severity_by_age["DriverAgeGroup"]
    .value_counts()
    .sort_index()
)

In [ ]:
severity_by_age_summary = (
    severity_by_age.groupby(
        "DriverAgeGroup",
        observed=True
    )
    .agg(
        ClaimCount=("ClaimAmount", "count"),
        MeanSeverity=("ClaimAmount", "mean"),
        MedianSeverity=("ClaimAmount", "median"),
        StdSeverity=("ClaimAmount", "std"),
        MaxSeverity=("ClaimAmount", "max")
    )
)

severity_by_age_summary.round(2)

In [ ]:
extreme_claims_by_age = []

for age, group in severity_by_age.groupby(
    "DriverAgeGroup",
    observed=True
):
    cutoff = group["ClaimAmount"].quantile(0.99)

    top_claims = group[
        group["ClaimAmount"] >= cutoff
    ]

    total_cost = group["ClaimAmount"].sum()
    top_cost = top_claims["ClaimAmount"].sum()

    extreme_claims_by_age.append({
        "DriverAgeGroup": age,
        "ClaimCount": len(group),
        "99thPercentile": cutoff,
        "Top1PercentCount": len(top_claims),
        "TotalClaimCost": total_cost,
        "Top1PercentCost": top_cost,
        "Top1PercentCostShare": 100 * top_cost / total_cost,
        "MeanExcludingTop1Percent": group.loc[
            group["ClaimAmount"] < cutoff,
            "ClaimAmount"
        ].mean()
    })

extreme_claims_by_age = pd.DataFrame(
    extreme_claims_by_age
).set_index("DriverAgeGroup")

extreme_claims_by_age.round(2)

In [ ]:
youngest_claims = (
    severity_by_age[
        severity_by_age["DriverAgeGroup"] == "18-24"
    ]
    .sort_values("ClaimAmount", ascending=False)
    .head(10)
)

youngest_claims[
    ["IDpol", "ClaimAmount"]
].reset_index(drop=True).round(2)

In [ ]:
youngest_total = severity_by_age.loc[
    severity_by_age["DriverAgeGroup"] == "18-24",
    "ClaimAmount"
].sum()

youngest_top5 = youngest_claims.head(5)["ClaimAmount"].sum()

print("Largest five claims:", round(youngest_top5, 2))
print(
    "Share of age-group claim cost:",
    round(100 * youngest_top5 / youngest_total, 2),
    "%"
)

In [ ]:
severity_comparison = pd.DataFrame({
    "Original Mean": severity_by_age_summary["MeanSeverity"],
    "Mean Excluding Top 1%":
        extreme_claims_by_age["MeanExcludingTop1Percent"]
})

severity_comparison.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Average Claim Severity by Driver Age")
plt.ylabel("Average Claim Amount ($)")
plt.xlabel("Driver Age Group")
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
severity_by_vehicle_age = sev.merge(
    model_data[["IDpol", "VehicleAgeGroup"]].drop_duplicates(),
    on="IDpol",
    how="inner"
)

vehicle_severity_summary = (
    severity_by_vehicle_age
    .groupby("VehicleAgeGroup", observed=True)
    .agg(
        ClaimCount=("ClaimAmount", "count"),
        MeanSeverity=("ClaimAmount", "mean"),
        MedianSeverity=("ClaimAmount", "median"),
        StdSeverity=("ClaimAmount", "std"),
        MaxSeverity=("ClaimAmount", "max")
    )
)

vehicle_severity_summary.round(2)

In [ ]:
extreme_claims_by_vehicle_age = []

for age, group in severity_by_vehicle_age.groupby(
    "VehicleAgeGroup",
    observed=True
):
    cutoff = group["ClaimAmount"].quantile(0.99)

    top_claims = group[
        group["ClaimAmount"] >= cutoff
    ]

    total_cost = group["ClaimAmount"].sum()
    top_cost = top_claims["ClaimAmount"].sum()

    extreme_claims_by_vehicle_age.append({
        "VehicleAgeGroup": age,
        "ClaimCount": len(group),
        "99thPercentile": cutoff,
        "Top1PercentCount": len(top_claims),
        "Top1PercentCostShare": 100 * top_cost / total_cost,
        "MeanExcludingTop1Percent": group.loc[
            group["ClaimAmount"] < cutoff,
            "ClaimAmount"
        ].mean()
    })

extreme_claims_by_vehicle_age = (
    pd.DataFrame(extreme_claims_by_vehicle_age)
    .set_index("VehicleAgeGroup")
)

extreme_claims_by_vehicle_age.round(2)

In [ ]:
severity_by_density = sev.merge(
    model_data[["IDpol", "Density"]].drop_duplicates(),
    on="IDpol",
    how="inner"
)

severity_by_density["DensityGroup"] = pd.qcut(
    severity_by_density["Density"],
    q=5,
    labels=["Q1 - Lowest", "Q2", "Q3", "Q4", "Q5 - Highest"]
)

density_severity_summary = (
    severity_by_density
    .groupby("DensityGroup", observed=True)
    .agg(
        ClaimCount=("ClaimAmount", "count"),
        MeanSeverity=("ClaimAmount", "mean"),
        MedianSeverity=("ClaimAmount", "median"),
        StdSeverity=("ClaimAmount", "std"),
        MaxSeverity=("ClaimAmount", "max")
    )
)

density_severity_summary.round(2)

In [ ]:
extreme_claims_by_density = []

for density, group in severity_by_density.groupby(
    "DensityGroup",
    observed=True
):
    cutoff = group["ClaimAmount"].quantile(0.99)

    top_claims = group[
        group["ClaimAmount"] >= cutoff
    ]

    total_cost = group["ClaimAmount"].sum()
    top_cost = top_claims["ClaimAmount"].sum()

    extreme_claims_by_density.append({
        "DensityGroup": density,
        "ClaimCount": len(group),
        "99thPercentile": cutoff,
        "Top1PercentCount": len(top_claims),
        "Top1PercentCostShare": 100 * top_cost / total_cost,
        "MeanExcludingTop1Percent": group.loc[
            group["ClaimAmount"] < cutoff,
            "ClaimAmount"
        ].mean()
    })

extreme_claims_by_density = (
    pd.DataFrame(extreme_claims_by_density)
    .set_index("DensityGroup")
)

extreme_claims_by_density.round(2)

In [ ]:
severity_model_data = severity_by_age.merge(
    model_data[
        ["IDpol", "VehicleAgeGroup", "LogDensity"]
    ].drop_duplicates(),
    on="IDpol",
    how="inner"
)

print("Severity modeling records:", len(severity_model_data))
print(
    "Missing values:",
    severity_model_data[
        ["ClaimAmount", "DriverAgeGroup",
         "VehicleAgeGroup", "LogDensity"]
    ].isna().sum().sum()
)

severity_model_data.head()

In [ ]:
gamma_formula = (
    "ClaimAmount ~ "
    "C(DriverAgeGroup) + "
    "C(VehicleAgeGroup) + "
    "LogDensity"
)

gamma_model = smf.glm(
    formula=gamma_formula,
    data=severity_model_data,
    family=sm.families.Gamma(
        link=sm.families.links.Log()
    )
).fit()

print(gamma_model.summary())

In [ ]:
gamma_effects = pd.DataFrame({
    "Coefficient": gamma_model.params,
    "P_Value": gamma_model.pvalues,
    "Lower_CI": gamma_model.conf_int()[0],
    "Upper_CI": gamma_model.conf_int()[1]
})

gamma_effects["RelativeSeverity"] = np.exp(
    gamma_effects["Coefficient"]
)

gamma_effects["LowerRelative"] = np.exp(
    gamma_effects["Lower_CI"]
)

gamma_effects["UpperRelative"] = np.exp(
    gamma_effects["Upper_CI"]
)

gamma_effects.round(4)

In [ ]:
lognormal_model = smf.ols(
    formula=(
        "np.log(ClaimAmount) ~ "
        "C(DriverAgeGroup) + "
        "C(VehicleAgeGroup) + "
        "LogDensity"
    ),
    data=severity_model_data
).fit()

print(lognormal_model.summary())

In [ ]:
from sklearn.model_selection import train_test_split

severity_train, severity_test = train_test_split(
    severity_model_data,
    test_size=0.20,
    random_state=42
)

print("Training claims:", len(severity_train))
print("Testing claims:", len(severity_test))

In [ ]:
gamma_train = smf.glm(
    formula=gamma_formula,
    data=severity_train,
    family=sm.families.Gamma(
        link=sm.families.links.Log()
    )
).fit()

lognormal_train = smf.ols(
    formula=(
        "np.log(ClaimAmount) ~ "
        "C(DriverAgeGroup) + "
        "C(VehicleAgeGroup) + "
        "LogDensity"
    ),
    data=severity_train
).fit()

print("Gamma model fitted.")
print("Lognormal model fitted.")

In [ ]:
# Gamma predictions (dollars)
gamma_test_pred = gamma_train.predict(severity_test)

# Lognormal predictions (convert log predictions to dollars)
smearing_factor = np.mean(np.exp(lognormal_train.resid))

lognormal_test_pred = (
    np.exp(lognormal_train.predict(severity_test))
    * smearing_factor
)

print("Gamma predictions:")
print(gamma_test_pred.describe())

print("\nLognormal predictions:")
print(lognormal_test_pred.describe())

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

actual = severity_test["ClaimAmount"]

results = pd.DataFrame({
    "Model": ["Gamma", "Lognormal"],
    "MAE": [
        mean_absolute_error(actual, gamma_test_pred),
        mean_absolute_error(actual, lognormal_test_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(actual, gamma_test_pred)),
        np.sqrt(mean_squared_error(actual, lognormal_test_pred))
    ]
})

print(results.round(2))

In [ ]:
comparison = pd.DataFrame({
    "Actual": actual,
    "Gamma_Pred": gamma_test_pred,
    "Lognormal_Pred": lognormal_test_pred
})

comparison["ActualGroup"] = pd.qcut(
    comparison["Actual"],
    q=5,
    labels=["Lowest 20%", "20-40%", "40-60%", "60-80%", "Highest 20%"]
)

comparison["Gamma_Error"] = (
    comparison["Actual"] - comparison["Gamma_Pred"]
).abs()

comparison["Lognormal_Error"] = (
    comparison["Actual"] - comparison["Lognormal_Pred"]
).abs()

error_by_group = comparison.groupby(
    "ActualGroup", observed=False
)[["Gamma_Error", "Lognormal_Error"]].mean()

print(error_by_group.round(2))

In [ ]:
large_loss_threshold = severity_train["ClaimAmount"].quantile(0.99)

print("Large-loss threshold: $", round(large_loss_threshold, 2))

print(
    "Training claims above threshold:",
    (severity_train["ClaimAmount"] > large_loss_threshold).sum()
)

In [ ]:
large_claims = severity_train[
    severity_train["ClaimAmount"] > large_loss_threshold
]

ordinary_claims = severity_train[
    severity_train["ClaimAmount"] <= large_loss_threshold
]

print("Ordinary claims:", len(ordinary_claims))
print("Large claims:", len(large_claims))

print("\nAverage ordinary claim: $", round(
    ordinary_claims["ClaimAmount"].mean(), 2
))

print("Average large claim: $", round(
    large_claims["ClaimAmount"].mean(), 2
))

print("\nLarge-loss share of total claim cost:",
      round(
          large_claims["ClaimAmount"].sum()
          / severity_train["ClaimAmount"].sum() * 100,
          2
      ), "%")

In [ ]:
severity_train["LargeLoss"] = (
    severity_train["ClaimAmount"] > large_loss_threshold
).astype(int)

large_loss_model = smf.logit(
    formula=(
        "LargeLoss ~ "
        "C(DriverAgeGroup) + "
        "C(VehicleAgeGroup) + "Warning: Maximum number of iterations has been exceeded.
         Current function value: 0.055524
         Iterations: 35
                           Logit Regression Results
==============================================================================
Dep. Variable:              LargeLoss   No. Observations:                21155
Model:                          Logit   Df Residuals:                    21143
Method:                           MLE   Df Model:                           11
Date:                Sun, 01 Nov 2026   Pseudo R-squ.:                 0.01025
Time:                        18:22:32   Log-Likelihood:                -1174.6
converged:                      False   LL-Null:                       -1186.8
Covariance Type:            nonrobust   LLR p-value:                   0.01144
===============================================================================================
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -3.7414      0.353    -10.611      0.000      -4.432      -3.050
C(DriverAgeGroup)[T.25-34]     -0.6342      0.262     -2.421      0.015      -1.148      -0.121
C(DriverAgeGroup)[T.35-44]     -0.6195      0.249     -2.485      0.013      -1.108      -0.131
C(DriverAgeGroup)[T.45-54]     -0.5232      0.241     -2.169      0.030      -0.996      -0.050
C(DriverAgeGroup)[T.55-64]     -1.0190      0.310     -3.285      0.001      -1.627      -0.411
C(DriverAgeGroup)[T.65+]       -0.1651      0.267     -0.619      0.536      -0.688       0.358
C(VehicleAgeGroup)[T.2-5]      -0.1915      0.210     -0.913      0.361      -0.603       0.220
C(VehicleAgeGroup)[T.6-10]     -0.4657      0.218     -2.133      0.033      -0.894      -0.038
C(VehicleAgeGroup)[T.11-15]    -0.1846      0.222     -0.833      0.405      -0.619       0.250
C(VehicleAgeGroup)[T.16-20]    -0.5520      0.376     -1.469      0.142      -1.288       0.184
C(VehicleAgeGroup)[T.21+]     -16.6824   2808.391     -0.006      0.995   -5521.027    5487.662
LogDensity                     -0.0111      0.038     -0.293      0.769      -0.085       0.063
===============================================================================================

/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


        "LogDensity"
    ),
    data=severity_train
).fit()

print(large_loss_model.summary())

In [ ]:
print("Large losses by driver age:")
print(
    severity_train.groupby("DriverAgeGroup", observed=False)["LargeLoss"]
    .agg(["count", "sum"])
)

print("\nLarge losses by vehicle age:")
print(
    severity_train.groupby("VehicleAgeGroup", observed=False)["LargeLoss"]
    .agg(["count", "sum"])
)

In [ ]:
severity_train["VehicleAgeGroup2"] = (
    severity_train["VehicleAgeGroup"]
    .replace({
        "16-20": "16+",
        "21+": "16+"
    })
)

severity_test["VehicleAgeGroup2"] = (
    severity_test["VehicleAgeGroup"]
    .replace({
        "16-20": "16+",
        "21+": "16+"
    })
)

print(
    severity_train.groupby("VehicleAgeGroup2", observed=False)["LargeLoss"]
    .agg(["count", "sum"])
)

In [ ]:
large_loss_model = smf.logit(
    formula=(
        "LargeLoss ~ "
        "C(DriverAgeGroup) + "
        "C(VehicleAgeGroup2) + "
        "LogDensity"
    ),
    data=severity_train
).fit(maxiter=200)

print(large_loss_model.summary())

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

large_loss_test_pred = large_loss_model.predict(severity_test)

actual_large_loss = (
    severity_test["ClaimAmount"] > large_loss_threshold
).astype(int)

print("Test large-loss claims:", actual_large_loss.sum())
print("Test ordinary claims:", (actual_large_loss == 0).sum())

print("\nROC-AUC:",
      round(roc_auc_score(actual_large_loss, large_loss_test_pred), 4))

print("Average Precision:",
      round(average_precision_score(actual_large_loss, large_loss_test_pred), 4))

In [ ]:
print(
    severity_test.assign(LargeLoss=actual_large_loss)
    .groupby("DriverAgeGroup", observed=False)["LargeLoss"]
    .agg(["count", "sum", "mean"])
)

In [ ]:
large_loss_comparison = severity_test.copy()

large_loss_comparison["ActualLargeLoss"] = actual_large_loss
large_loss_comparison["PredictedLargeLoss"] = large_loss_test_pred

age_comparison = large_loss_comparison.groupby(
    "DriverAgeGroup", observed=False
).agg(
    ClaimCount=("ActualLargeLoss", "count"),
    ActualLargeLosses=("ActualLargeLoss", "sum"),
    ActualRate=("ActualLargeLoss", "mean"),
    PredictedRate=("PredictedLargeLoss", "mean")
)

age_comparison["ActualRate"] *= 100
age_comparison["PredictedRate"] *= 100

print(age_comparison.round(3))

In [ ]:
for name in [
    "freq", "model_data", "model_sample",
    "final_formula", "final_model",
    "severity_model_data", "severity_train"
]:
    print(f"{name}: {'Available' if name in globals() else 'Missing'}")

In [ ]:
final_formula = (
    "ClaimNb ~ C(DriverAgeGroup) "
    "+ C(VehicleAgeGroup) "
    "+ LogDensity "
    "+ BonusMalus "
    "+ C(VehPower) "
    "+ C(VehBrand) "
    "+ C(VehGas)"
)

final_model = smf.glm(
    formula=final_formula,
    data=model_sample,
    family=sm.families.Poisson(),
    offset=np.log(model_sample["Exposure"])
).fit(cov_type="HC0")

print(final_model.summary())

In [ ]:
for name in [
    "gamma_train",
    "gamma_formula",
    "severity_train",
    "severity_test",
    "model_sample"
]:
    print(f"{name}: {'Available' if name in globals() else 'Missing'}")

In [ ]:
# Predict expected claims over each policy's exposure period
predicted_claims = final_model.predict(
    model_sample,
    offset=np.log(model_sample["Exposure"])
)

# Convert to annual claim frequency
predicted_frequency = (
    predicted_claims / model_sample["Exposure"]
)

# Predict average claim severity
predicted_severity = gamma_train.predict(model_sample)

# Calculate predicted annual pure premium
predicted_pure_premium = (
    predicted_frequency * predicted_severity
)

print("Predicted annual pure premium:")
print(predicted_pure_premium.describe().round(2))

print("\nAverage predicted frequency:",
      round(predicted_frequency.mean(), 4))

print("Average predicted severity: $",
      round(predicted_severity.mean(), 2))

print("Average predicted pure premium: $",
      round(predicted_pure_premium.mean(), 2))

In [ ]:
# Start with the policies in our frequency modeling sample
comparison = model_sample[
    ["IDpol", "Exposure"]
].copy()

# Calculate actual claim costs for each policy
actual_costs = (
    severity_model_data
    .groupby("IDpol")["ClaimAmount"]
    .sum()
    .rename("ActualClaimCost")
)

comparison = comparison.merge(
    actual_costs,
    on="IDpol",
    how="left"
)

comparison["ActualClaimCost"] = (
    comparison["ActualClaimCost"].fillna(0)
)

# Add predicted values
comparison["PredictedFrequency"] = predicted_frequency
comparison["PredictedSeverity"] = predicted_severity
comparison["PredictedPurePremium"] = predicted_pure_premium

# Calculate portfolio-level annualized costs
total_exposure = comparison["Exposure"].sum()

actual_pure_premium = (
    comparison["ActualClaimCost"].sum() / total_exposure
)

predicted_pure_premium = (
    (comparison["PredictedPurePremium"] * comparison["Exposure"]).sum()
    / total_exposure
)

print(f"Total exposure: {total_exposure:,.2f} policy-years")
print(f"Actual portfolio pure premium: ${actual_pure_premium:,.2f}")
print(f"Predicted portfolio pure premium: ${predicted_pure_premium:,.2f}")

print(
    "Difference: "
    f"{(predicted_pure_premium / actual_pure_premium - 1) * 100:.2f}%"
)

In [ ]:
print("Number of policies:", len(model_sample))
print("Frequency predictions:", len(predicted_frequency))
print("Severity predictions:", len(predicted_severity))
print("Pure premium predictions:", len(predicted_pure_premium))
print("Comparison rows:", len(comparison))

print("\nAverage predicted frequency:",
      predicted_frequency.mean())

print("Average predicted severity:",
      predicted_severity.mean())

print("Average predicted pure premium:",
      predicted_pure_premium.mean())

print("\nExposure-weighted predicted pure premium:",
      (predicted_pure_premium * model_sample["Exposure"]).sum()
      / model_sample["Exposure"].sum())

In [ ]:
# Restore policy-level predictions
predicted_pure_premium = (
    predicted_frequency * predicted_severity
)

print("Number of policies:", len(model_sample))
print("Frequency predictions:", len(predicted_frequency))
print("Severity predictions:", len(predicted_severity))
print("Pure premium predictions:", len(predicted_pure_premium))

print("\nUnweighted average:",
      round(predicted_pure_premium.mean(), 2))

print("Exposure-weighted average:",
      round(
          (predicted_pure_premium * model_sample["Exposure"]).sum()
          / model_sample["Exposure"].sum(),
          2
      ))

In [ ]:
comparison = model_sample[
    ["IDpol", "Exposure", "ClaimNb"]
].copy().reset_index(drop=True)

# Add predictions by position, avoiding index alignment issues
comparison["PredictedFrequency"] = predicted_frequency.to_numpy()
comparison["PredictedSeverity"] = predicted_severity.to_numpy()
comparison["PredictedPurePremium"] = predicted_pure_premium.to_numpy()

# Add actual severity costs
actual_costs = (
    severity_model_data
    .groupby("IDpol")["ClaimAmount"]
    .sum()
    .rename("ActualClaimCost")
)

comparison = comparison.merge(
    actual_costs,
    on="IDpol",
    how="left"
)

comparison["ActualClaimCost"] = (
    comparison["ActualClaimCost"].fillna(0)
)

# Portfolio-level comparison
total_exposure = comparison["Exposure"].sum()

actual_pp = (
    comparison["ActualClaimCost"].sum() / total_exposure
)

predicted_pp = (
    (comparison["PredictedPurePremium"] * comparison["Exposure"]).sum()
    / total_exposure
)

print(f"Actual pure premium: ${actual_pp:,.2f}")
print(f"Predicted pure premium: ${predicted_pp:,.2f}")
print(f"Difference: {(predicted_pp / actual_pp - 1) * 100:.2f}%")

In [ ]:
age_comparison = comparison.merge(
    model_sample[["IDpol", "DriverAgeGroup"]],
    on="IDpol",
    how="left"
)

age_results = age_comparison.groupby(
    "DriverAgeGroup",
    observed=True
).apply(
    lambda x: pd.Series({
        "Policies": len(x),
        "Exposure": x["Exposure"].sum(),
        "ActualPurePremium": (
            x["ActualClaimCost"].sum() / x["Exposure"].sum()
        ),
        "PredictedPurePremium": (
            (x["PredictedPurePremium"] * x["Exposure"]).sum()
            / x["Exposure"].sum()
        )
    }),
    include_groups=False
)

age_results["DifferencePct"] = (
    (age_results["PredictedPurePremium"]
     / age_results["ActualPurePremium"] - 1) * 100
)

print(age_results.round(2))

In [ ]:
age_component_results = age_comparison.groupby(
    "DriverAgeGroup",
    observed=True
).apply(
    lambda x: pd.Series({
        "ActualFrequency": (
            x["ClaimNb"].sum() / x["Exposure"].sum()
        ),
        "PredictedFrequency": (
            (x["PredictedFrequency"] * x["Exposure"]).sum()
            / x["Exposure"].sum()
        ),
        "ActualSeverity": (
            x["ActualClaimCost"].sum() / x["ClaimNb"].sum()
            if x["ClaimNb"].sum() > 0 else np.nan
        ),
        "PredictedSeverity": (
            (x["PredictedSeverity"] * x["Exposure"]).sum()
            / (x["PredictedFrequency"] * x["Exposure"]).sum()
        )
    }),
    include_groups=False
)

print(age_component_results.round(3))

In [ ]:
age_component_results = age_comparison.groupby(
    "DriverAgeGroup",
    observed=True
).apply(
    lambda x: pd.Series({
        "ActualFrequency": (
            x["ClaimNb"].sum() / x["Exposure"].sum()
        ),
        "PredictedFrequency": (
            (x["PredictedFrequency"] * x["Exposure"]).sum()
            / x["Exposure"].sum()
        ),
        "ActualSeverity": (
            x["ActualClaimCost"].sum()
            / x["ClaimNb"].sum()
            if x["ClaimNb"].sum() > 0 else np.nan
        ),
        "PredictedSeverity": (
            (x["PredictedSeverity"]
             * x["PredictedFrequency"]
             * x["Exposure"]).sum()
            /
            (x["PredictedFrequency"] * x["Exposure"]).sum()
        )
    }),
    include_groups=False
)

print(age_component_results.round(2))

In [ ]:
age_component_results["SeverityDifferencePct"] = (
    (
        age_component_results["PredictedSeverity"]
        / age_component_results["ActualSeverity"]
    ) - 1
) * 100

print(
    age_component_results[
        ["ActualSeverity", "PredictedSeverity", "SeverityDifferencePct"]
    ].round(2)
)

In [ ]:
# Calculate the 99th percentile using training data only
severity_cutoff = severity_train["ClaimAmount"].quantile(0.99)

# Retain claims at or below the cutoff
severity_train_trimmed = severity_train[
    severity_train["ClaimAmount"] <= severity_cutoff
].copy()

print(f"99th percentile cutoff: ${severity_cutoff:,.2f}")
print(f"Original training claims: {len(severity_train):,}")
print(f"Trimmed training claims: {len(severity_train_trimmed):,}")
print(f"Removed claims: {len(severity_train) - len(severity_train_trimmed):,}")

In [ ]:
gamma_trimmed = smf.glm(
    formula=gamma_formula,
    data=severity_train_trimmed,
    family=sm.families.Gamma(
        link=sm.families.links.Log()
    )
).fit()

print(gamma_trimmed.summary())

In [ ]:
# Predict severity on the original, untrimmed test set
gamma_trimmed_pred = gamma_trimmed.predict(severity_test)

# Evaluate against actual test claims
trimmed_mae = np.mean(
    np.abs(severity_test["ClaimAmount"] - gamma_trimmed_pred)
)

trimmed_rmse = np.sqrt(
    np.mean(
        (severity_test["ClaimAmount"] - gamma_trimmed_pred) ** 2
    )
)

print(f"Trimmed Gamma MAE:  ${trimmed_mae:,.2f}")
print(f"Trimmed Gamma RMSE: ${trimmed_rmse:,.2f}")

In [ ]:
# Predictions from both models on the same test observations
gamma_original_pred = gamma_train.predict(severity_test)

actual = severity_test["ClaimAmount"]

# Define claim-size groups using the training-set threshold
claim_groups = pd.cut(
    actual,
    bins=[-np.inf, large_loss_threshold, np.inf],
    labels=["Ordinary Claims", "Large Claims"]
)

results = pd.DataFrame({
    "Actual": actual,
    "OriginalPrediction": gamma_original_pred,
    "TrimmedPrediction": gamma_trimmed_pred,
    "ClaimGroup": claim_groups
})

group_results = results.groupby(
    "ClaimGroup",
    observed=True
).apply(
    lambda x: pd.Series({
        "ClaimCount": len(x),
        "OriginalMAE": np.mean(
            np.abs(x["Actual"] - x["OriginalPrediction"])
        ),
        "TrimmedMAE": np.mean(
            np.abs(x["Actual"] - x["TrimmedPrediction"])
        ),
        "OriginalRMSE": np.sqrt(
            np.mean((x["Actual"] - x["OriginalPrediction"]) ** 2)
        ),
        "TrimmedRMSE": np.sqrt(
            np.mean((x["Actual"] - x["TrimmedPrediction"]) ** 2)
        )
    }),
    include_groups=False
)

print(group_results.round(2))

In [ ]:
def large_loss_summary(data, threshold):
    large_claims = data[
        data["ClaimAmount"] > threshold
    ]["ClaimAmount"]

    return pd.Series({
        "Count": len(large_claims),
        "Mean": large_claims.mean(),
        "Median": large_claims.median(),
        "Maximum": large_claims.max(),
        "TotalCost": large_claims.sum()
    })

large_loss_comparison = pd.DataFrame({
    "Training": large_loss_summary(
        severity_train, large_loss_threshold
    ),
    "Testing": large_loss_summary(
        severity_test, large_loss_threshold
    )
})

print(large_loss_comparison.round(2))

In [ ]:
def large_loss_proportions(data, threshold):
    total_cost = data["ClaimAmount"].sum()

    large_claims = data[
        data["ClaimAmount"] > threshold
    ]["ClaimAmount"]

    return pd.Series({
        "TotalClaims": len(data),
        "LargeClaims": len(large_claims),
        "LargeClaimPercentage": (
            len(large_claims) / len(data) * 100
        ),
        "TotalClaimCost": total_cost,
        "LargeClaimCost": large_claims.sum(),
        "LargeClaimCostPercentage": (
            large_claims.sum() / total_cost * 100
        )
    })

large_loss_shares = pd.DataFrame({
    "Training": large_loss_proportions(
        severity_train, large_loss_threshold
    ),
    "Testing": large_loss_proportions(
        severity_test, large_loss_threshold
    )
})

print(large_loss_shares.round(2))

In [ ]:
# Separate training claims
ordinary_train = severity_train[
    severity_train["ClaimAmount"] <= large_loss_threshold
]["ClaimAmount"]

large_train = severity_train[
    severity_train["ClaimAmount"] > large_loss_threshold
]["ClaimAmount"]

# Calculate component probabilities and averages
p_ordinary = len(ordinary_train) / len(severity_train)
p_large = len(large_train) / len(severity_train)

mean_ordinary = ordinary_train.mean()
mean_large = large_train.mean()

# Reconstruct overall average severity
combined_severity = (
    p_ordinary * mean_ordinary
    + p_large * mean_large
)

print(f"Ordinary claim probability: {p_ordinary:.4%}")
print(f"Large claim probability: {p_large:.4%}")
print(f"Average ordinary severity: ${mean_ordinary:,.2f}")
print(f"Average large severity: ${mean_large:,.2f}")
print(f"Combined expected severity: ${combined_severity:,.2f}")
print(f"Actual training average severity: ${severity_train['ClaimAmount'].mean():,.2f}")

In [ ]:
vehicle_age_comparison = comparison.merge(
    model_sample[["IDpol", "VehicleAgeGroup"]],
    on="IDpol",
    how="left"
)

vehicle_age_results = vehicle_age_comparison.groupby(
    "VehicleAgeGroup",
    observed=True
).apply(
    lambda x: pd.Series({
        "Policies": len(x),
        "Exposure": x["Exposure"].sum(),
        "ActualPurePremium": (
            x["ActualClaimCost"].sum() / x["Exposure"].sum()
        ),
        "PredictedPurePremium": (
            (x["PredictedPurePremium"] * x["Exposure"]).sum()
            / x["Exposure"].sum()
        )
    }),
    include_groups=False
)

vehicle_age_results["DifferencePct"] = (
    (vehicle_age_results["PredictedPurePremium"]
     / vehicle_age_results["ActualPurePremium"] - 1) * 100
)

print(vehicle_age_results.round(2))

In [ ]:
density_comparison = comparison.merge(
    model_sample[["IDpol", "Density"]],
    on="IDpol",
    how="left"
)

density_comparison["DensityGroup"] = pd.qcut(
    density_comparison["Density"],
    q=5,
    labels=["Q1 - Lowest", "Q2", "Q3", "Q4", "Q5 - Highest"]
)

density_results = density_comparison.groupby(
    "DensityGroup",
    observed=True
).apply(
    lambda x: pd.Series({
        "Policies": len(x),
        "Exposure": x["Exposure"].sum(),
        "ActualPurePremium": (
            x["ActualClaimCost"].sum() / x["Exposure"].sum()
        ),
        "PredictedPurePremium": (
            (x["PredictedPurePremium"] * x["Exposure"]).sum()
            / x["Exposure"].sum()
        )
    }),
    include_groups=False
)

density_results["DifferencePct"] = (
    (density_results["PredictedPurePremium"]
     / density_results["ActualPurePremium"] - 1) * 100
)

print(density_results.round(2))

In [ ]:
from sklearn.model_selection import train_test_split

policy_train, policy_test = train_test_split(
    model_sample,
    test_size=0.20,
    random_state=42
)

train_ids = set(policy_train["IDpol"])
test_ids = set(policy_test["IDpol"])

print("Training policies:", len(policy_train))
print("Testing policies:", len(policy_test))
print("Overlapping policy IDs:", len(train_ids.intersection(test_ids)))

In [ ]:
frequency_model = smf.glm(
    formula=final_formula,
    data=policy_train,
    family=sm.families.Poisson(),
    offset=np.log(policy_train["Exposure"])
).fit(cov_type="HC0")

print(frequency_model.summary())

In [ ]:
test_frequency = frequency_model.predict(
    policy_test,
    offset=np.zeros(len(policy_test))
)

policy_test = policy_test.copy()
policy_test["PredictedFrequency"] = test_frequency

print("Actual claim frequency:",
      policy_test["ClaimNb"].sum() / policy_test["Exposure"].sum())

print("Predicted claim frequency:",
      (policy_test["PredictedFrequency"] *
       policy_test["Exposure"]).sum() / policy_test["Exposure"].sum())

In [ ]:
severity_train_policy = severity_model_data[
    severity_model_data["IDpol"].isin(train_ids)
].copy()

severity_test_policy = severity_model_data[
    severity_model_data["IDpol"].isin(test_ids)
].copy()

print("Training severity records:", len(severity_train_policy))
print("Testing severity records:", len(severity_test_policy))

print(
    "Overlapping policy IDs:",
    len(
        set(severity_train_policy["IDpol"])
        .intersection(set(severity_test_policy["IDpol"]))
    )
)

In [ ]:
gamma_policy_model = smf.glm(
    formula=gamma_formula,
    data=severity_train_policy,
    family=sm.families.Gamma(
        link=sm.families.links.Log()
    )
).fit()

print(gamma_policy_model.summary())

In [ ]:
pricing_test = policy_test.copy()

# Predict average severity per claim
pricing_test["PredictedSeverity"] = (
    gamma_policy_model.predict(pricing_test)
)

# Calculate annual pure premium
pricing_test["PredictedPurePremium"] = (
    pricing_test["PredictedFrequency"]
    * pricing_test["PredictedSeverity"]
)

# Actual claim costs for held-out policies
actual_test_costs = (
    severity_test_policy.groupby("IDpol")["ClaimAmount"]
    .sum()
    .rename("ActualClaimCost")
)

pricing_test = pricing_test.merge(
    actual_test_costs,
    on="IDpol",
    how="left"
)

pricing_test["ActualClaimCost"] = (
    pricing_test["ActualClaimCost"].fillna(0)
)

# Portfolio-level comparison
actual_pure_premium = (
    pricing_test["ActualClaimCost"].sum()
    / pricing_test["Exposure"].sum()
)

predicted_pure_premium = (
    (pricing_test["PredictedPurePremium"]
     * pricing_test["Exposure"]).sum()
    / pricing_test["Exposure"].sum()
)

print(f"Actual pure premium: ${actual_pure_premium:,.2f}")
print(f"Predicted pure premium: ${predicted_pure_premium:,.2f}")
print(
    "Percentage difference: "
    f"{(predicted_pure_premium / actual_pure_premium - 1) * 100:.2f}%"
)

In [ ]:
severity_test_policy = severity_test_policy.copy()

severity_test_policy["PredictedSeverity"] = (
    gamma_policy_model.predict(severity_test_policy)
)

mae = np.mean(
    np.abs(
        severity_test_policy["ClaimAmount"]
        - severity_test_policy["PredictedSeverity"]
    )
)

rmse = np.sqrt(
    np.mean(
        (severity_test_policy["ClaimAmount"]
         - severity_test_policy["PredictedSeverity"]) ** 2
    )
)

actual_mean = severity_test_policy["ClaimAmount"].mean()
predicted_mean = severity_test_policy["PredictedSeverity"].mean()

print(f"Actual mean severity: ${actual_mean:,.2f}")
print(f"Predicted mean severity: ${predicted_mean:,.2f}")
print(f"MAE: ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

In [ ]:
severity_test_policy["SeverityGroup"] = pd.qcut(
    severity_test_policy["ClaimAmount"],
    q=5,
    labels=[
        "Lowest 20%",
        "20-40%",
        "40-60%",
        "60-80%",
        "Highest 20%"
    ]
)

severity_group_results = severity_test_policy.groupby(
    "SeverityGroup",
    observed=True
).apply(
    lambda x: pd.Series({
        "ClaimCount": len(x),
        "ActualMean": x["ClaimAmount"].mean(),
        "PredictedMean": x["PredictedSeverity"].mean(),
        "MAE": np.abs(
            x["ClaimAmount"] - x["PredictedSeverity"]
        ).mean()
    }),
    include_groups=False
)

severity_group_results["DifferencePct"] = (
    (severity_group_results["PredictedMean"]
     / severity_group_results["ActualMean"] - 1) * 100
)

print(severity_group_results.round(2))

In [ ]:
severity_cost_results = severity_test_policy.groupby(
    "SeverityGroup",
    observed=True
).apply(
    lambda x: pd.Series({
        "ClaimCount": len(x),
        "ActualTotalCost": x["ClaimAmount"].sum(),
        "PredictedTotalCost": x["PredictedSeverity"].sum()
    }),
    include_groups=False
)

severity_cost_results["ActualCostShare"] = (
    severity_cost_results["ActualTotalCost"]
    / severity_test_policy["ClaimAmount"].sum() * 100
)

severity_cost_results["PredictedCostShare"] = (
    severity_cost_results["PredictedTotalCost"]
    / severity_test_policy["PredictedSeverity"].sum() * 100
)

print(severity_cost_results.round(2))